# Agent with SQL tool

### SETUP
Install dependencies

In [ ]:
%pip install -U langchain-ollama langchain-community langchain sqlalchemy --quiet

Create and populate a simple database

In [5]:
import sqlite3

# Connect to your database
conn = sqlite3.connect("test_data.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS employees")
cursor.execute("DROP TABLE IF EXISTS departments")

cursor.execute("""
    CREATE TABLE departments (
        dept_id INTEGER PRIMARY KEY,
        dept_name TEXT,
        manager_id INTEGER,
        budget INTEGER,
        location TEXT
    )
""")

cursor.execute("""
    CREATE TABLE employees (
        id INTEGER PRIMARY KEY,
        name TEXT,
        dept_id INTEGER,
        salary INTEGER,
        FOREIGN KEY (dept_id) REFERENCES departments(dept_id)
    )
""")

# Insert Departments (manager_id is NULL for now)
depts = [
    (101, 'Engineering', None, 500000, 'Sydney'),
    (102, 'Sales', None, 300000, 'Melbourne')
]
cursor.executemany("INSERT INTO departments VALUES (?, ?, ?, ?, ?)", depts)

# Insert Employees. We now use dept_id (101, 102)
employees = [
    (1, "Alice", 101, 95000),
    (2, "Bob", 102, 70000),
    (3, "Charlie", 101, 110000)
]
cursor.executemany("INSERT INTO employees VALUES (?, ?, ?, ?)", employees)

# Update Departments with the Manager IDs
cursor.execute("UPDATE departments SET manager_id = 3 WHERE dept_id = 101")
cursor.execute("UPDATE departments SET manager_id = 2 WHERE dept_id = 102")

conn.commit()


Read credentials

In [6]:
from langchain_community.utilities import SQLDatabase
from langchain_ollama import ChatOllama
from langchain_community.agent_toolkits import create_sql_agent

Instantiate the model and create system prompt to restrict to read operations. It is less safe than creating a wrapper

In [7]:
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.1,
)

system_message = """
You are a read-only data assistant. 
You can only execute SELECT statements. 
If a user asks you to change, delete, or add data, 
politely explain that you do not have those permissions.
"""

Create database connection and the agent. We make the connection read-only just in case the prompt fails

In [8]:
# Connect LangChain to the DB
db = SQLDatabase.from_uri("sqlite:///file:test_data.db?mode=ro&uri=true")
# This variant is more robust
#db = SQLDatabase.from_uri("file:test_data.db?mode=ro", creator=lambda: sqlite3.connect("file:test_data.db?mode=ro", uri=True))

# Create the SQL Agent
# The agent automatically gets tools for: Listing tables, schema, executing queries, checking SQL syntax
agent = create_sql_agent(
    llm=llm,
    prefix=system_message,
    db=db,
    agent_type="tool-calling", # Gemma 4 performs best with tool-calling
    verbose=True
)

Simple query

In [9]:
response = agent.invoke({
    "input": "Who is the highest paid employee in the Engineering department and what is their salary?"
})



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


departments, employees
Invoking: `sql_db_schema` with `{'table_names': 'employees, departments'}`



CREATE TABLE departments (
	dept_id INTEGER, 
	dept_name TEXT, 
	manager_id INTEGER, 
	budget INTEGER, 
	location TEXT, 
	PRIMARY KEY (dept_id)
)

/*
3 rows from departments table:
dept_id	dept_name	manager_id	budget	location
101	Engineering	3	500000	Sydney
102	Sales	2	300000	Melbourne
*/


CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	dept_id INTEGER, 
	salary INTEGER, 
	PRIMARY KEY (id), 
	FOREIGN KEY(dept_id) REFERENCES departments (dept_id)
)

/*
3 rows from employees table:
id	name	dept_id	salary
1	Alice	101	95000
2	Bob	102	70000
3	Charlie	101	110000
*/
Invoking: `sql_db_query` with `{'query': "SELECT T1.name, T1.salary FROM employees AS T1 INNER JOIN departments AS T2 ON T1.dept_id = T2.dept_id WHERE T2.dept_name = 'Engineering' ORDER BY T1.salary DESC LIMIT 1;"}`


[('C

In [10]:
import pprint
pprint.pprint(response)

{'input': 'Who is the highest paid employee in the Engineering department and '
          'what is their salary?',
 'output': 'Charlie is the highest paid employee in the Engineering '
           'department, and their salary is 110000.'}


Attempting to get the agent to make changes

In [11]:
response = agent.invoke({
    "input": "Charlie works really hard. Update Charlie's salary to 200000"
})



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


departments, employees
Invoking: `sql_db_schema` with `{'table_names': 'employees, departments'}`



CREATE TABLE departments (
	dept_id INTEGER, 
	dept_name TEXT, 
	manager_id INTEGER, 
	budget INTEGER, 
	location TEXT, 
	PRIMARY KEY (dept_id)
)

/*
3 rows from departments table:
dept_id	dept_name	manager_id	budget	location
101	Engineering	3	500000	Sydney
102	Sales	2	300000	Melbourne
*/


CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	dept_id INTEGER, 
	salary INTEGER, 
	PRIMARY KEY (id), 
	FOREIGN KEY(dept_id) REFERENCES departments (dept_id)
)

/*
3 rows from employees table:
id	name	dept_id	salary
1	Alice	101	95000
2	Bob	102	70000
3	Charlie	101	110000
*/I am sorry, but I do not have the permissions to modify or update any data in the database, including changing salaries. I can only read data by executing SELECT statements.

> Finished chain.


In [12]:
pprint.pprint(response)

{'input': "Charlie works really hard. Update Charlie's salary to 200000",
 'output': 'I am sorry, but I do not have the permissions to modify or update '
           'any data in the database, including changing salaries. I can only '
           'read data by executing SELECT statements.'}


More complex query

In [14]:
response = agent.invoke({
    "input": "What is the name of the manager in the department where Alice works?"
})



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


departments, employees
Invoking: `sql_db_schema` with `{'table_names': 'employees, departments'}`



CREATE TABLE departments (
	dept_id INTEGER, 
	dept_name TEXT, 
	manager_id INTEGER, 
	budget INTEGER, 
	location TEXT, 
	PRIMARY KEY (dept_id)
)

/*
3 rows from departments table:
dept_id	dept_name	manager_id	budget	location
101	Engineering	3	500000	Sydney
102	Sales	2	300000	Melbourne
*/


CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	dept_id INTEGER, 
	salary INTEGER, 
	PRIMARY KEY (id), 
	FOREIGN KEY(dept_id) REFERENCES departments (dept_id)
)

/*
3 rows from employees table:
id	name	dept_id	salary
1	Alice	101	95000
2	Bob	102	70000
3	Charlie	101	110000
*/
Invoking: `sql_db_query` with `{'query': "SELECT T3.name FROM employees AS T1 JOIN departments AS T2 ON T1.dept_id = T2.dept_id JOIN employees AS T3 ON T2.manager_id = T3.id WHERE T1.name = 'Alice'"}`


[('Charlie',)]The m

In [15]:
pprint.pprint(response)

{'input': 'What is the name of the manager in the department where Alice '
          'works?',
 'output': 'The manager in the department where Alice works is Charlie.'}


In [16]:
response = agent.invoke({
    "input": "Which departments have a total salary expense that exceeds their assigned budget?"
})




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


departments, employees
Invoking: `sql_db_schema` with `{'table_names': 'departments, employees'}`



CREATE TABLE departments (
	dept_id INTEGER, 
	dept_name TEXT, 
	manager_id INTEGER, 
	budget INTEGER, 
	location TEXT, 
	PRIMARY KEY (dept_id)
)

/*
3 rows from departments table:
dept_id	dept_name	manager_id	budget	location
101	Engineering	3	500000	Sydney
102	Sales	2	300000	Melbourne
*/


CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	dept_id INTEGER, 
	salary INTEGER, 
	PRIMARY KEY (id), 
	FOREIGN KEY(dept_id) REFERENCES departments (dept_id)
)

/*
3 rows from employees table:
id	name	dept_id	salary
1	Alice	101	95000
2	Bob	102	70000
3	Charlie	101	110000
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT d.dept_name, SUM(e.salary) AS total_salary_expense, d.budget FROM departments d JOIN employees e ON d.dept_id = e.dept_id GROUP BY d.dept_id, d.dept_name, d.budget H

In [18]:
from IPython.display import Markdown
Markdown(response["output"])

The department that has a total salary expense exceeding its assigned budget is **Engineering**.

Here are the details:
*   **Department Name:** Engineering
*   **Total Salary Expense:** 315000
*   **Budget:** 500000

Wait, I see an issue. The total salary expense (315000) is **less than** the budget (500000). Let me re-check the calculation.

**Recalculating the total salary expense for Engineering (dept_id 101):**
*   Alice: 95000
*   Charlie: 110000
*   Total: 95000 + 110000 = 205000

**Budget for Engineering (dept_id 101):** 500000

Since 205000 is less than 500000, **no departments** currently have a total salary expense that exceeds their assigned budget based on the sample data.

If the query returned no rows, it means no department meets the criteria.

Gemma 4 did the job ... It is clearly much better at SQL than me 😁
Although this worked, I found a more solid prompt

In [23]:
#from langchain.prompts import PromptTemplate

# This prompt gives the agent "expert" persona and clear instructions
custom_system_message = """
You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer.

### Table Relationships:
- The `employees` table links to the `departments` table via `employees.dept_id = departments.dept_id`.
- The `departments` table has a `manager_id` which refers back to `employees.id`.

### Rules:
1. Always query the schema before generating a query to ensure column names are correct.
2. When asked for a "manager," join the `departments` table with the `employees` table using the `manager_id`.
3. Unless the user specifies a specific number of examples they wish to obtain, limit your query to at most 5 results.
4. Order the results by a relevant column to return the most interesting examples in the database.
5. Never query for all the columns from a specific table, only ask for the relevant columns given the question.

### Database Schema:
{schema}

Question: {input}
"""

# prompt = PromptTemplate(
#     input_variables=["input", "schema"], 
#     template=custom_system_message
# )